# AIC2026 — ma hoa anh bang model thu hai (ViT-gopt-16-SigLIP2-384)

Huong dan day du: `docs/17_kaggle_encode_va_caption.md`.

## Settings BAT BUOC truoc khi chay

| | |
| --- | --- |
| Accelerator | **GPU T4 x2** |
| Internet | **ON** |
| Input | `aic2026-index` + 9 dataset anh |

⚠️ **DUNG chon P100.** P100 la compute capability **sm_60**; torch tren Kaggle
chi ho tro **sm_70 tro len**. Luot chay 30/08 dung P100 va chet voi
`CUDA error: no kernel image is available for execution on the device` —
sau khi da tai xong 7,49 GB trong so. T4 la sm_75, chay duoc.

## 1. Ma nguon

Clone vao `/tmp` chu **khong** vao `/kaggle/working`: moi thu trong
`/kaggle/working` deu tro thanh **output cua notebook**, nen clone vao do la
ca cay repo (ke ca `.git` 10 MB) bi dinh kem moi phien ban.

In [ ]:
!rm -rf /tmp/repo
!git clone -q -b giai-doan-0 https://github.com/QuocKhanhDev-it/AIC_2026_FirstDance.git /tmp/repo
%cd /tmp/repo
!pip -q install open_clip_torch pandas pyarrow

## 2. Chot GPU — chay TRUOC khi tai 7,49 GB trong so

Model gopt nang **7,49 GB**. Phat hien GPU khong dung duoc *sau* khi tai xong
la mat trang vai phut quota cho khong.

In [ ]:
import torch
assert torch.cuda.is_available(), "khong co GPU — kiem Settings > Accelerator"
ten = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
ho_tro = torch.cuda.get_arch_list()
print(f"GPU: {ten}  compute capability sm_{cc[0]}{cc[1]}")
print("torch build cho:", ho_tro)
assert f"sm_{cc[0]}{cc[1]}" in ho_tro, (
    f"{ten} (sm_{cc[0]}{cc[1]}) KHONG nam trong build torch nay {ho_tro}. "
    "Doi Accelerator sang T4 x2 (sm_75). P100 la sm_60, khong chay duoc.")
print("OK")

## 3. Chep `index/` tu Private Dataset

TIM file chu KHONG doan duong dan. Luot chay 30/08 cho thay Kaggle mount o
`/kaggle/input/datasets/<user>/<slug>/`, **khong** phai `/kaggle/input/<slug>/`.

In [ ]:
import glob, shutil, os, pathlib
print("co trong /kaggle/input:", os.listdir('/kaggle/input'))
pathlib.Path('index').mkdir(exist_ok=True)
for ten in ('master.parquet', 'clip.npy', 'trung_lap.parquet'):
    hit = glob.glob(f'/kaggle/input/**/{ten}', recursive=True)
    if hit:
        shutil.copy(hit[0], f'index/{ten}')
        print(f"  {ten:20} <- {hit[0]}")
    else:
        print(f"  {ten:20} KHONG THAY (--kiem-lech-hang se khong chay duoc)")
assert os.path.exists('index/master.parquet'), "thieu master.parquet"

## 4. Va duong dan — BUOC BAT BUOC, va la buoc de quen nhat

`kf_path` trong `master.parquet` la duong dan tuyet doi cua may dung index
(`D:\Project\...`). Tren Kaggle no khong ton tai. Bo qua buoc nay thi
`08_encode.py` thay **khong co anh nao** va ghi ra mot ma tran toan so 0 —
**khong bao loi gi**.

In [ ]:
!python scripts/12_va_duong_dan.py --roots /kaggle/input --ghi

### Ham `chay()` — vi sao khong dung `!lenh`

`!lenh` that bai **KHONG** lam dung notebook: cell van tinh la chay xong, cac
cell sau chay tiep, va Kaggle danh dau ca phien la **COMPLETE**. Luot 30/08
"thanh cong" trong khi moi buoc encode deu chet.

`chay()` nem `RuntimeError` khi ma thoat khac 0, de hong o dau thi dung o do.

In [ ]:
import subprocess, sys

def chay(lenh):
    print("$", lenh, flush=True)
    p = subprocess.run(lenh, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"ma thoat {p.returncode}: {lenh}")
    return p.stdout

### Chot chan — GOI TEN nhom con thieu

Mot con so tong khong du. Ngay 30/08 mot may chay ra `88.635 / 97.731` va nhin
qua thi van "gan dung" — thuc ra **thieu han nhom L22** vi quen `Add Input`.
Neu encode tiep thi 9.096 dong lang le thanh vector 0, va khong co gi bao.

Cell nay doi chieu **tung nhom** voi so mong doi va goi ten cai thieu.

In [ ]:
import pandas as pd

# So keyframe moi nhom — thuoc tinh cua kho BTC, khong doi theo may.
# L26 chua ai co anh nen khong nam trong bang nay.
MONG_DOI = {'L21': 7800, 'L22': 9096, 'L23': 2326, 'L24': 6781, 'L25': 37445,
            'L27': 4914, 'L28': 10683, 'L29': 10771, 'L30': 7915}

m = pd.read_parquet('index/master.parquet')
co = m[m.kf_path.notna()].video_id.str[:3].value_counts().to_dict()

thieu, lech = [], []
for nhom, can in sorted(MONG_DOI.items()):
    duoc = int(co.get(nhom, 0))
    dau = "OK " if duoc == can else ("THIEU HAN" if duoc == 0 else "LECH")
    print(f"  {nhom}  {duoc:>7,} / {can:>7,}  {dau}")
    if duoc == 0:
        thieu.append(nhom)
    elif duoc != can:
        lech.append((nhom, duoc, can))

tong = sum(co.get(k, 0) for k in MONG_DOI)
print(f"\n  tong  {tong:,} / {sum(MONG_DOI.values()):,}")

assert not thieu, (
    f"CHUA Add Input cho nhom: {', '.join(thieu)}. "
    "Them vao Input roi chay lai tu cell nay.")
assert not lech, f"So anh lech o {lech} — dataset chua giai nen xong?"
print("\nDu ca 9 nhom.")

## 5. Bac 0 — 10 video (2.634 anh), chi de soat

⚠️ **DUNG dung `--videos 100`.** No chon 10 video MOI NHOM L va encode tron ven
moi keyframe cua chung: **25.824 anh = 26% toan bo cong viec**, khong con la
phep thu nua. `--videos 10` cho 2.634 anh, va van du **253 keyframe trung lap**
cho `--kiem-lech-hang` (phep kiem lay mau 200).

Doc 3 thu trong log truoc khi leo tiep:

| doc gi | phai la |
| --- | --- |
| so chieu | **1536** — ra 1152 la dang chay nham model cu |
| `--kiem-lech-hang` | dat |
| **anh/giay** | ghi lai, ca ke hoach dua vao con so nay |

`--workers 4` vi Kaggle cap 4 vCPU. Tran VRAM thi ha `--batch` xuong 16.

In [ ]:
chay("python scripts/08_encode.py --model ViT-gopt-16-SigLIP2-384 --pretrained webli "
     "--videos 10 --workers 4 --batch 32 --out /kaggle/working/thu.npy")

Lech hang la loi nguy hiem nhat o day: moi vector ve sai anh, cosine van dep,
moi kiem tra cau truc van xanh, va diem thi tut ma khong ai biet vi sao.

Phep kiem can `index/clip.npy` + `index/trung_lap.parquet` (buoc 3 da chep).

In [ ]:
chay("python scripts/08_encode.py --kiem-lech-hang /kaggle/working/thu.npy")

### Gio thuc te cho phan con lai

Chua co so do cho model nay tren phan cung Kaggle. No ~1,1 ty tham so, gap gan
ba lan SO400M — dung suy ra tu con so 23,9 anh/giay do tren may khac, gpu khac.

Ra duoi ~4 anh/giay thi tong vuot 7 gio. Luc do doi sang
`ViT-L-16-SigLIP2-384` (1024 chieu, nhe hon nhieu): van la model thu hai doc
lap, va mot ma tran chay xong dang gia hon mot ma tran chay do.

In [ ]:
ANH_MOI_GIAY = 0      # <-- dien so THAT doc duoc o bac 0
if ANH_MOI_GIAY:
    print(f"97.731 anh con lai -> {97731 / ANH_MOI_GIAY / 3600:.1f} gio")
    print("moi phien Kaggle toi da 12 gio, quota 30 gio/tuan")

## 6. Phan cua toi — doi DUNG MOT SO o cell duoi

97.731 anh chia 6 phan, moi phan ~16.290 anh, lech nhau **28 anh**.

| phan | video | anh |
| ---: | ---: | ---: |
| 1 | 63 | 16.299 |
| 2 | 63 | 16.303 |
| 3 | 63 | 16.303 |
| 4 | 62 | 16.276 |
| 5 | 62 | 16.275 |
| 6 | 62 | 16.275 |

**Khong ai phat file danh sach cho ai.** Notebook goi thang
`scripts/47_chia_viec_encode.py` de tu tinh — thuat toan tat dinh, cung
`master.parquet` thi cung ket qua, nen 6 nguoi chay ra 6 phan roi nhau ma
khong can trao doi gi. Phat file qua chat la duong de dung nham ban cu.

Neu lam MOT MINH thi chay lan luot `PHAN_CUA_TOI = 1..6`.

In [ ]:
PHAN_CUA_TOI = 1        # <-- DOI SO NAY, tu 1 den 6. Khong doi gi khac.
SO_NGUOI = 6

import importlib.util, pandas as pd
_s = importlib.util.spec_from_file_location(
    "chia_viec", "scripts/47_chia_viec_encode.py")
_cv = importlib.util.module_from_spec(_s); _s.loader.exec_module(_cv)

m = pd.read_parquet('index/master.parquet')
phan, tai = _cv.chia(m, SO_NGUOI)
v = phan[PHAN_CUA_TOI - 1]

TEN_DS = f'phan_{PHAN_CUA_TOI}.txt'
RA = f'/kaggle/working/clip_gopt_phan{PHAN_CUA_TOI}.npy'
open(TEN_DS, 'w').write('\n'.join(v) + '\n')

print(f"phan {PHAN_CUA_TOI}/{SO_NGUOI}: {len(v)} video, "
      f"{tai[PHAN_CUA_TOI - 1]:,} anh")
print(f"  cac phan khac: {tai}")
print(f"  ra: {RA}")

In [ ]:
chay(f"python scripts/08_encode.py --model ViT-gopt-16-SigLIP2-384 --pretrained webli "
     f"--chi-video {TEN_DS} --workers 4 --batch 32 --out {RA}")

## 7. Cache truy van — PHAI sinh, va sinh o DAY

`index/truy_van.npz` hien tai ma hoa bang thap van ban SO400M, **1152 chieu**.
Model nay **1536 chieu** — khong co cach nao dung lai cache cu. Cho nay khong
hong im lang (`KenhAnhCache` so so chieu roi dung han), nhung dung de no bat.

Sinh trong **cung notebook nay** vi model 7,49 GB da tai san o day. Tach ra
notebook khac la tai lai lan nua, ton quota cho mot viec chi mat vai phut.

`--matrix` la du: script doc ten model va so chieu tu sidecar `.json` ma
`08_encode.py` vua ghi.

In [ ]:
import glob, shutil, os
sc = sorted(glob.glob('/kaggle/working/clip_gopt_phan*.npy'))
print("ma tran da co:", [os.path.basename(x) for x in sc])
# sidecar phai nam CANH ma tran trong index/ thi 25_ moi doc duoc ten model
for f in sc + [x[:-4] + '.json' for x in sc]:
    if os.path.exists(f):
        shutil.copy(f, 'index/' + os.path.basename(f))

In [ ]:
TEN_MA_TRAN = f'clip_gopt_phan{PHAN_CUA_TOI}.npy'
chay(f"python scripts/25_ma_hoa_truy_van.py --matrix {TEN_MA_TRAN} "
     f"--ra index/truy_van_gopt.npz --tap-dev --fp16")
shutil.copy('index/truy_van_gopt.npz', '/kaggle/working/')

## 8. Soat truoc khi tai ve

Tai `clip_gopt_phan*.npy` + file `.json` cung ten, va `truy_van_gopt.npz`.

> ⚠️ **KHONG tai `master.parquet` tu Kaggle ve.** File do da bi buoc 4 va thanh
> duong dan `/kaggle/input/...`. De no len may local la moi thu doc anh chet
> hang loat. Chi tai `.npy`, `.json`, `.npz`.

> Cell nay chi de NHIN. No boc `try` quanh tung file va khong bao gio duoc
> phep lam hong ca luot chay: luot 30/08 encode xong het, sinh cache xong het,
> roi **chet o dung cell tong ket** vi mot file `.npy` khong phai 2 chieu —
> `np.abs(a).sum(1)` nem `AxisError`. Ket qua khong mat gi, nhung Kaggle danh
> dau ca phien la ERROR.

In [ ]:
import numpy as np, glob, os
for f in sorted(glob.glob('/kaggle/working/*.np[yz]')):
    ten = os.path.basename(f)
    try:
        if f.endswith('.npz'):
            z = np.load(f, allow_pickle=False)
            print(f"{ten:28} npz: {', '.join(z.files)}")
            if 'vec' in z:
                print(f"{'':28} vec {z['vec'].shape} {z['vec'].dtype}")
            continue
        a = np.load(f, mmap_mode='r')
        if a.ndim != 2:
            print(f"{ten:28} {str(a.shape):18} {a.dtype}  (khong phai ma tran)")
            continue
        print(f"{ten:28} {str(a.shape):18} {a.dtype}  "
              f"co vector: {int((np.abs(a).sum(1) > 0).sum()):,}")
    except Exception as e:
        print(f"{ten:28} doc khong duoc: {type(e).__name__}: {str(e)[:60]}")

print()
!ls -la /kaggle/working/